In [15]:
from pydantic import BaseModel
import uuid

class Task(BaseModel):
    """Task class to hold task information."""
    task_id: str | None = None
    task_status: str = "initiated"
    task_description: str | None = None
    task_name: str
    task_parameters: Dict[str, Any] | None = None
    task_state: Dict[str, Any] | None = None

    def __init__(
            self, 
            task_name: str, 
            task_description = None, 
            task_id: str = None, 
            task_parameters: Dict[str, Any] = None,
            **kwargs
        ) -> None:
        if not task_id:
            task_id = str(uuid.uuid4())
        task_parameters["ext_args"]=kwargs
        task_state=task_parameters["ext_args"].get("task_state",None)
        super().__init__(
            task_id=task_id,
            task_status="pending" if task_id else "initiated",
            task_name=task_name,
            task_description=task_description,
            task_parameters=task_parameters,
        )
    
    def to_dict(self) -> Dict[str, Any]:
        return {
            "task_id": self.task_id,
            "task_status": self.task_status,
            "task_name": self.task_name,
            "task_description": self.task_description,
            "task_parameters": self.task_parameters,
            "task_state": self.task_state
        }
    
    def __str__(self) -> str:
        return f"Task: {self.task_name} ({self.task_id}) - {self.task_status}"


In [18]:
task_dict = {"task_name":"extract_data","task_parameters":{"source_content_id":"fc_12345"}}

In [19]:
task = Task(**task_dict)

In [20]:
print(task.task_parameters)

{'source_content_id': 'fc_12345', 'ext_args': {}}


In [1]:
from mcp.server.fastmcp import FastMCP
from autogen_ext.agents.probill import ProbillPlanAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_core.models import ModelFamily
from autogen_agentchat.base import TaskResult


model_info = {
    "vision": False,
    "function_calling": True,
    "json_output": True,
    "family": ModelFamily.R1
}

model_client=OllamaChatCompletionClient(
    model="qwq", 
    host="http://10.0.40.49:11434",
    model_info=model_info,  
    model_client_stream=True,
)

plan_agent = ProbillPlanAgent("proboll_planner", model_client=model_client)
async def run_stream():
    async for result in plan_agent.run_stream(task="How are you?"):
        if isinstance(result, TaskResult):
            for msg in result.messages:
                print(msg.source)
                print(msg.content)
        else:
            print(result.content)

await run_stream()

How are you?
<think>
Okay, the user asked "How are you?" which is a common greeting. I need to respond appropriately.

First, since I'm an AI, I don't have feelings, so I should mention that. But I should also be friendly and offer help. Let me check if there's any code needed here. The user didn't ask for a specific task that requires coding, just a general question about my status. 

The instructions say to use code blocks only when necessary. Since this is a simple greeting, no code is needed. I can respond directly with a friendly message. Make sure to keep it natural and helpful.
</think>

I'm just a large language model, so I don't have feelings or emotions. However, I'm here to help you with any questions or tasks you might have! How can I assist you today?
user
How are you?
proboll_planner
<think>
Okay, the user asked "How are you?" which is a common greeting. I need to respond appropriately.

First, since I'm an AI, I don't have feelings, so I should mention that. But I should

In [2]:
_component_plan_agent = plan_agent.dump_component()
component_json = _component_plan_agent.model_dump()

AttributeError: 'ProbillPlanAgent' object has no attribute '_participants'

In [9]:
print(component_json)

{
  "provider": "autogen_ext.agents.probill.ProbillPlanAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "An agent, used by Probill that provides plaaning service.",
  "label": "ProbillPlanAgent",
  "config": {
    "name": "proboll_planner",
    "model_client": {
      "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for Ollama hosted models.",
      "label": "OllamaChatCompletionClient",
      "config": {
        "model": "qwq",
        "host": "http://10.0.40.49:11434",
        "follow_redirects": true
      }
    },
    "tools": [],
    "model_context": {
      "provider": "autogen_core.model_context.UnboundedChatCompletionContext",
      "component_type": "chat_completion_context",
      "version": 1,
      "component_version": 1,
      "description": "An unbounded chat completion context tha

In [13]:
component_json.get("config", {}).get("model_client",{}).get("config")["model_info"]=model_client.model_info

In [15]:
import json
print(json.dumps(component_json, indent=2))

{
  "provider": "autogen_ext.agents.probill.ProbillPlanAgent",
  "component_type": "agent",
  "version": 1,
  "component_version": 1,
  "description": "An agent, used by Probill that provides plaaning service.",
  "label": "ProbillPlanAgent",
  "config": {
    "name": "proboll_planner",
    "model_client": {
      "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
      "component_type": "model",
      "version": 1,
      "component_version": 1,
      "description": "Chat completion client for Ollama hosted models.",
      "label": "OllamaChatCompletionClient",
      "config": {
        "model": "qwq",
        "host": "http://10.0.40.49:11434",
        "follow_redirects": true,
        "model_info": {
          "vision": false,
          "function_calling": true,
          "json_output": true,
          "family": "r1"
        }
      }
    },
    "tools": [],
    "model_context": {
      "provider": "autogen_core.model_context.UnboundedChatCompletionContext",
      "co

In [ ]:
model_cpn = model_client.dump_component()

print(model_cpn.model_dump_json(indent=2))

{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}


In [5]:
model_client.model_info

{'vision': False,
 'function_calling': True,
 'json_output': True,
 'family': 'r1'}

In [20]:
temp = """{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "model_info": {
        "vision": false,
        "function_calling": true,
        "json_output": true,
        "family": "r1"
    },
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}"""

temp_json = json.loads(temp)

In [21]:
print(json.dumps(temp_json, indent=2))

{
  "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
  "component_type": "model",
  "version": 1,
  "component_version": 1,
  "description": "Chat completion client for Ollama hosted models.",
  "label": "OllamaChatCompletionClient",
  "config": {
    "model": "qwq",
    "model_info": {
      "vision": false,
      "function_calling": true,
      "json_output": true,
      "family": "r1"
    },
    "host": "http://10.0.40.49:11434",
    "follow_redirects": true
  }
}


In [ ]:
from autogen_agentchat.teams import MagenticOneGroupChat